# Ridge regression baseline

This notebook builds our simplest model. For each of the 28 continuous sensors, we train a Ridge regression that predicts that sensor from the other 35 features in the same hour. If the prediction error is big, the hour looks unusual, and that is how we find attacks.

- Data used: the files in `data/processed/`: `train_scaled.csv`, `val_scaled.csv`, `cal_scaled.csv` and `dataset2_eval.csv` (Dataset 2 with the labels from the 7 attack windows).
- The Test set is not used. It stays untouched until all choices and thresholds are frozen.
- The full write-up is in `docs/ridge.md`. The notes here are short.

Steps:
1. Load the processed data
2. Try one sensor first (`L_T1`)
3. Fit one model for each of the 28 continuous sensors
4. Pick the best alpha for each sensor
5. Turn the errors into one score per hour, and set the threshold on Calibration
6. Raise the threshold to stay inside the false alarm budget
7. Score Dataset 2
8. Look at the false alarms on Dataset 2
9. Try again without `P_J422`
10. Save the results

## 1. Load the processed data

Why: every model in the project should use the same split and the same scaling, so we load the saved files from the pipeline (`docs/pipeline.md`) instead of the raw CSVs.

What the code does: loads Training (`train`), Validation (`val`), Calibration (`cal`) and Dataset 2 (`d2`) from `../data/processed/`. It builds the lists of continuous columns (`cont_cols`), on/off columns (`onoff_cols`) and all features (`features`), and prints the shapes and the list sizes.

In [1]:
import pandas as pd

folder = "../data/processed"

train = pd.read_csv(f"{folder}/train_scaled.csv", parse_dates=["DATETIME"])
val = pd.read_csv(f"{folder}/val_scaled.csv", parse_dates=["DATETIME"])
cal = pd.read_csv(f"{folder}/cal_scaled.csv", parse_dates=["DATETIME"])
d2 = pd.read_csv(f"{folder}/dataset2_eval.csv", parse_dates=["DATETIME"])

cont_cols = [c for c in train.columns if c.startswith(("L_", "F_", "P_"))]
onoff_cols = [c for c in train.columns if c.startswith("S_")]
features = cont_cols + onoff_cols

for name, df in [("Training", train), ("Validation", val), ("Calibration", cal), ("Dataset 2", d2)]:
    print(name, df.shape)

print(len(cont_cols), len(onoff_cols), len(features))

Training (5256, 38)
Validation (1728, 38)
Calibration (1729, 38)
Dataset 2 (4177, 40)
28 8 36


**What this shows:**
- Training is 5,256 × 38, Validation is 1,728 × 38 and Calibration is 1,729 × 38.
- Dataset 2 is 4,177 × 40. It has 2 extra columns: `label` (1 inside an attack window, 0 outside) and `attack_id` (1 to 7, or 0 outside attacks).
- 28 continuous + 8 on/off = 36 features.

## 2. Try one sensor first: `L_T1`

Why: before fitting 28 models, we check that the idea works on one sensor. We pick `L_T1` (tank 1 level), because attacks 3 and 4 overflow tank 1.

What the code does: trains a Ridge model (`alpha=1.0`) that predicts `L_T1` from the other 35 features, using Training only. It then computes the error (real value minus prediction) on Training, Validation and Dataset 2. It prints the mean absolute error on each, splits Dataset 2 into outside and inside attacks, and prints the mean error for each attack (`attack_id` 0 means outside attacks).

In [2]:
from sklearn.linear_model import Ridge

target = "L_T1"
inputs = [c for c in features if c != target]

model = Ridge(alpha=1.0)
model.fit(train[inputs], train[target])

err_train = train[target] - model.predict(train[inputs])
err_val = val[target] - model.predict(val[inputs])
err_d2 = d2[target] - model.predict(d2[inputs])

print("Training:", round(err_train.abs().mean(), 4))
print("Validation:", round(err_val.abs().mean(), 4))
print("Dataset 2, outside attacks:", round(err_d2[d2["label"] == 0].abs().mean(), 4))
print("Dataset 2, inside attacks:", round(err_d2[d2["label"] == 1].abs().mean(), 4))

print(err_d2.abs().groupby(d2["attack_id"]).mean().round(4))

Training: 0.0713
Validation: 0.0716
Dataset 2, outside attacks: 0.074
Dataset 2, inside attacks: 0.2547
attack_id
0    0.0740
1    0.1203
2    0.4671
3    0.4014
4    0.6264
5    0.0761
6    0.0966
7    0.1044
Name: L_T1, dtype: float64


**What this shows:**
- Training error 0.0713 and Validation error 0.0716 are almost the same, so the model is not overfitting. Dataset 2 outside attacks (0.074) is also about the same.
- Inside attacks the error is much bigger: 0.2547, more than 3 times the normal error.
- Attacks 3 (0.4014) and 4 (0.6264) are clearly caught. These attacks change tank 1 directly.
- Attack 2 (0.4671) also shows up. Attack 2 fakes the pump 10/11 readings, and those readings are inputs to the `L_T1` model. So the model gets wrong inputs and makes a bad prediction.
- Attack 7 (0.1044) is mostly missed on `L_T1`. In attack 7, `L_T1` and pumps 1/2 were replayed together, so the inputs and the target still fit each other.

Caution: a big error on a sensor doesn't always mean that sensor is attacked. Attack 2 does not touch `L_T1`, but `L_T1` still has a big error because some of its inputs are faked.

## 3. One model for each of the 28 continuous sensors

Why: one sensor can't catch every attack. Each attack changes different sensors, so we fit a model for every continuous sensor.

What the code does: for each sensor in `cont_cols`, trains a Ridge model (`alpha=1.0`) on Training that predicts it from the other 35 features, and stores it in `models`. It saves the errors for Training, Validation, Calibration and Dataset 2 (`err_train`, `err_val`, `err_cal`, `err_d2`, one column per sensor). It prints a table with the mean absolute error on Validation (`val`), Dataset 2 outside attacks (`d2_outside`) and inside attacks (`d2_inside`), and the ratio inside / outside. Last, it prints the sensor with the biggest mean error for each attack.

In [3]:
models = {}
err_train = pd.DataFrame(index=train.index)
err_val = pd.DataFrame(index=val.index)
err_cal = pd.DataFrame(index=cal.index)
err_d2 = pd.DataFrame(index=d2.index)

for target in cont_cols:
    inputs = [c for c in features if c != target]
    model = Ridge(alpha=1.0)
    model.fit(train[inputs], train[target])
    models[target] = model

    err_train[target] = train[target] - model.predict(train[inputs])
    err_val[target] = val[target] - model.predict(val[inputs])
    err_cal[target] = cal[target] - model.predict(cal[inputs])
    err_d2[target] = d2[target] - model.predict(d2[inputs])

summary = pd.DataFrame({
    "val": err_val.abs().mean(),
    "d2_outside": err_d2[d2["label"] == 0].abs().mean(),
    "d2_inside": err_d2[d2["label"] == 1].abs().mean(),
})
summary["ratio"] = summary["d2_inside"] / summary["d2_outside"]
print(summary.sort_values("ratio", ascending=False).round(3))

by_attack = err_d2.abs().groupby(d2["attack_id"]).mean()
print(pd.DataFrame({"top_sensor": by_attack.idxmax(axis=1), "error": by_attack.max(axis=1).round(3)}))

          val  d2_outside  d2_inside   ratio
F_PU6   0.002       0.002      0.069  32.472
F_PU11  0.001       0.002      0.021  13.625
F_PU7   0.007       0.007      0.085  12.802
P_J14   0.008       0.008      0.065   8.245
F_PU1   0.003       0.003      0.024   7.108
F_V2    0.011       0.011      0.069   6.177
P_J269  0.004       0.004      0.023   5.210
F_PU2   0.007       0.006      0.029   4.594
P_J415  0.014       0.014      0.061   4.310
P_J307  0.004       0.004      0.014   3.657
L_T1    0.072       0.074      0.255   3.442
P_J302  0.004       0.004      0.013   3.437
P_J289  0.002       0.002      0.007   3.177
P_J300  0.002       0.002      0.007   3.065
P_J317  0.019       0.019      0.048   2.532
F_PU10  0.007       0.007      0.016   2.377
L_T2    0.047       0.050      0.106   2.128
P_J422  0.003       0.003      0.005   2.048
L_T4    0.150       0.148      0.267   1.812
P_J306  0.015       0.015      0.026   1.762
P_J256  0.027       0.027      0.047   1.722
F_PU8   0.

**What this shows:**
- For every sensor, the Validation error and the Dataset 2 outside-attack error are about the same. So the models work just as well on new normal data.
- Tanks have much bigger normal errors than flows and pressures. For example `L_T4` is 0.150 on Validation, but `F_PU6` is 0.002. A tank level depends on what happened in the past hours (how much water came in and went out). Ridge only sees the current hour, so it can't know that.
- The biggest ratios are `F_PU6` (32.5), `F_PU11` (13.6) and `F_PU7` (12.8). Their normal errors are tiny, so attacks stand out a lot.
- Every attack has at least one sensor that reacts. The top sensor per attack:

| Attack | Top sensor | Mean error |
|---|---|---|
| 1 | `L_T7` | 0.238 |
| 2 | `L_T1` | 0.467 |
| 3 | `L_T1` | 0.401 |
| 4 | `L_T1` | 0.626 |
| 5 | `F_PU7` | 0.242 |
| 6 | `L_T4` | 0.485 |
| 7 | `L_T4` | 0.285 |
| outside attacks (0) | `L_T4` | 0.148 |

Caution: the raw errors can't be added up directly. A tank error of 0.15 is normal, but a flow error of 0.15 would be huge. If we just added them, the tanks would always win. Step 5 fixes this by dividing each sensor's error by its normal size.

## 4. Pick the best alpha for each sensor

Why: alpha is the Ridge penalty. A bigger alpha keeps the weights small. We pick it on Validation, never on Dataset 2. We first tried 0.01 to 100, and the best was at the smallest value, 0.01. So we extended the range down to 0.0001. The code now tries 0.0001, 0.001, 0.01, 0.1 and 1.

What the code does: for each alpha and each sensor, trains a model on Training and saves the mean absolute error on Validation in `val_err`. It picks the alpha with the smallest Validation error for each sensor (`best_alpha`), prints the table and how many sensors picked each alpha. Then it refits all 28 models with their own best alpha and recomputes all the errors (`err_train`, `err_val`, `err_cal`, `err_d2`). From here on, every step uses these new models.

In [5]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]
val_err = pd.DataFrame(index=cont_cols, columns=alphas, dtype=float)

for alpha in alphas:
    for target in cont_cols:
        inputs = [c for c in features if c != target]
        model = Ridge(alpha=alpha)
        model.fit(train[inputs], train[target])
        val_err.loc[target, alpha] = (val[target] - model.predict(val[inputs])).abs().mean()

best_alpha = val_err.idxmin(axis=1)

table = val_err.round(4)
table["best"] = best_alpha
print(table)
print(best_alpha.value_counts())

models = {}
for target in cont_cols:
    inputs = [c for c in features if c != target]
    model = Ridge(alpha=best_alpha[target])
    model.fit(train[inputs], train[target])
    models[target] = model

    err_train[target] = train[target] - model.predict(train[inputs])
    err_val[target] = val[target] - model.predict(val[inputs])
    err_cal[target] = cal[target] - model.predict(cal[inputs])
    err_d2[target] = d2[target] - model.predict(d2[inputs])

print(err_train.shape, err_val.shape, err_cal.shape, err_d2.shape)

        0.0001   0.001    0.01     0.1     1.0    best
L_T1    0.0602  0.0605  0.0609  0.0642  0.0716  0.0001
L_T2    0.0418  0.0424  0.0431  0.0446  0.0473  0.0001
L_T3    0.1229  0.1230  0.1230  0.1230  0.1230  0.0001
L_T4    0.1436  0.1437  0.1439  0.1454  0.1498  0.0001
L_T5    0.1328  0.1330  0.1332  0.1339  0.1434  0.0001
L_T6    0.1315  0.1319  0.1319  0.1318  0.1320  0.0001
L_T7    0.1044  0.1044  0.1044  0.1044  0.1045  0.1000
F_PU1   0.0002  0.0002  0.0004  0.0012  0.0034  0.0001
F_PU2   0.0011  0.0014  0.0034  0.0048  0.0065  0.0001
F_PU4   0.0084  0.0099  0.0111  0.0115  0.0117  0.0001
F_PU6   0.0001  0.0000  0.0001  0.0004  0.0020  0.0010
F_PU7   0.0054  0.0054  0.0055  0.0056  0.0066  0.0001
F_PU8   0.0079  0.0081  0.0081  0.0082  0.0088  0.0001
F_PU10  0.0050  0.0052  0.0053  0.0054  0.0068  0.0001
F_PU11  0.0002  0.0002  0.0001  0.0002  0.0015  0.0100
F_V2    0.0066  0.0067  0.0067  0.0070  0.0110  0.0001
P_J280  0.0845  0.0875  0.0894  0.0928  0.1060  0.0001
P_J269  0.

**What this shows:**
- Most sensors picked the smallest alpha: 23 of 28 picked 0.0001. Two picked 0.001 (`F_PU6`, `P_J269`), one picked 0.01 (`F_PU11`), and two picked 0.1 (`L_T7`, `P_J422`).
- Below 0.01 the errors barely change. For example `L_T1` is 0.0609 at 0.01 and 0.0602 at 0.0001. So Ridge needs very little penalty here, and the exact value doesn't matter much.
- The tuned alphas do help compared with alpha 1. `L_T1` goes from 0.0716 to 0.0602 on Validation.
- Each sensor uses its own best alpha. All four error tables have 28 columns, one per sensor.

## 5. One score per hour, and the first threshold

Why: we need one number per hour to decide alarm or no alarm. Raw errors have different sizes (step 3), so first we divide each sensor's error by its normal size. Then we take the average of the 3 sensors with the biggest scaled error in that hour. This is the same top-3 rule as the fusion in the proposal. An attack usually changes a few sensors, not all 28, so averaging all of them would hide it. Taking the top 3 instead of only the top 1 means one noisy sensor alone is less likely to cause an alarm.

What the code does: `normal_size` is the mean absolute Validation error of each sensor. `hourly_score` divides each error by `normal_size` and averages the top 3 values per hour. It scores Validation, Calibration and Dataset 2 (`score_val`, `score_cal`, `score_d2`). It sets the threshold to the 99th percentile of the Calibration scores and prints the number of alarm hours and alarm episodes in Calibration. An episode is a run of alarm hours in a row, counted once.

In [6]:
import numpy as np

normal_size = err_val.abs().mean()

def hourly_score(err):
    scaled = err.abs() / normal_size
    top3 = np.sort(scaled.values, axis=1)[:, -3:]
    return pd.Series(top3.mean(axis=1), index=err.index)

score_val = hourly_score(err_val)
score_cal = hourly_score(err_cal)
score_d2 = hourly_score(err_d2)

threshold = score_cal.quantile(0.99)
print("Threshold:", round(threshold, 3))

alarm_cal = score_cal > threshold
episodes_cal = (alarm_cal & ~alarm_cal.shift(1, fill_value=False)).sum()
print("Calibration alarm hours:", alarm_cal.sum(), "of", len(alarm_cal))
print("Calibration alarm episodes:", episodes_cal, "in", round(len(alarm_cal) / 168, 1), "weeks")

Threshold: 6.81
Calibration alarm hours: 18 of 1729
Calibration alarm episodes: 17 in 10.3 weeks


**What this shows:**
- The 99th percentile threshold is 6.81.
- Calibration has 18 alarm hours out of 1,729, in 17 separate episodes, over 10.3 weeks.

Caution: Calibration has no attacks, so every alarm there is a false alarm. 17 episodes in 10.3 weeks is more than 1 per week, which is over our false alarm budget. The next step fixes this.

## 6. Raise the threshold to stay inside the budget

Why: our budget is at most 1 false alarm episode per week on Calibration. With 10.3 weeks, that is 10 episodes. We also want to know which sensors cause the false alarms.

What the code does: first prints, for each Calibration alarm hour, the sensor with the biggest scaled error. `count_episodes` counts the alarm episodes. `max_episodes` is the number of full weeks in Calibration. Then it goes through the Calibration scores from the 99th percentile upwards, and picks the lowest threshold that gives `max_episodes` or fewer episodes. It prints the new threshold and the new alarm counts.

In [7]:
scaled_cal = err_cal.abs() / normal_size
print(scaled_cal[alarm_cal].idxmax(axis=1).value_counts())

def count_episodes(alarm):
    return (alarm & ~alarm.shift(1, fill_value=False)).sum()

max_episodes = int(len(score_cal) / 168)

candidates = np.sort(score_cal.unique())
candidates = candidates[candidates >= threshold]

for t in candidates:
    if count_episodes(score_cal > t) <= max_episodes:
        threshold = t
        break

alarm_cal = score_cal > threshold
print("Max episodes allowed:", max_episodes)
print("New threshold:", round(threshold, 3))
print("Calibration alarm hours:", alarm_cal.sum(), "of", len(alarm_cal))
print("Calibration alarm episodes:", count_episodes(alarm_cal))

P_J422    12
F_PU6      2
P_J317     1
P_J269     1
P_J306     1
F_PU10     1
Name: count, dtype: int64
Max episodes allowed: 10
New threshold: 7.276
Calibration alarm hours: 11 of 1729
Calibration alarm episodes: 10


**What this shows:**
- The false alarms at the old threshold came from: `P_J422` 12, `F_PU6` 2, and `P_J317`, `P_J269`, `P_J306` and `F_PU10` 1 each. So `P_J422` causes most of them.
- The budget is 10 episodes.
- The new threshold is 7.276. Calibration now has 11 alarm hours in 10 episodes, so it is inside the budget.
- This threshold is fixed from now on. Dataset 2 is not used to set it.

## 7. Score Dataset 2

Why: Dataset 2 has 7 labelled attacks, so we can see how well the model finds them, using the threshold from Calibration.

What the code does: `y` is the Dataset 2 label and `alarm_d2` is every hour with a score above the threshold. It prints ROC-AUC and PR-AUC (these use the score, no threshold), and precision, recall and F1 (these use the alarms). It counts the false alarm hours and episodes outside the attacks. Then for each attack it prints the number of hours, the number of alarm hours, whether it was detected (at least one alarm), and the delay (`delay_h`: hours from the start of the attack to the first alarm).

In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

y = d2["label"]
alarm_d2 = score_d2 > threshold

print("ROC-AUC:", round(roc_auc_score(y, score_d2), 3))
print("PR-AUC:", round(average_precision_score(y, score_d2), 3))
print("Precision:", round(precision_score(y, alarm_d2), 3))
print("Recall:", round(recall_score(y, alarm_d2), 3))
print("F1:", round(f1_score(y, alarm_d2), 3))

false_alarm = alarm_d2 & (y == 0)
print("False alarm hours:", false_alarm.sum(), "of", (y == 0).sum())
print("False alarm episodes:", count_episodes(false_alarm), "in", round((y == 0).sum() / 168, 1), "weeks")

rows = []
for a in range(1, 8):
    inside = d2["attack_id"] == a
    hits = alarm_d2[inside]
    first = hits.values.argmax() if hits.any() else None
    rows.append({"attack": a, "hours": inside.sum(), "alarm_hours": hits.sum(),
                 "detected": hits.any(), "delay_h": first})
print(pd.DataFrame(rows))

ROC-AUC: 0.866
PR-AUC: 0.741
Precision: 0.893
Recall: 0.61
F1: 0.725
False alarm hours: 36 of 3685
False alarm episodes: 32 in 21.9 weeks
   attack  hours  alarm_hours  detected  delay_h
0       1     50           29      True        0
1       2     24           19      True        0
2       3     60           10      True       43
3       4     94           86      True        0
4       5     60           60      True        0
5       6     94           70      True       10
6       7    110           26      True        5


**What this shows:**

| Metric | Dataset 2 |
|---|---|
| ROC-AUC | 0.866 |
| PR-AUC | 0.741 |
| Precision | 0.893 |
| Recall | 0.61 |
| F1 | 0.725 |
| False alarm hours | 36 of 3,685 |
| False alarm episodes | 32 in 21.9 weeks |

| Attack | Hours | Alarm hours | Detected | Delay (hours) |
|---|---|---|---|---|
| 1 | 50 | 29 | Yes | 0 |
| 2 | 24 | 19 | Yes | 0 |
| 3 | 60 | 10 | Yes | 43 |
| 4 | 94 | 86 | Yes | 0 |
| 5 | 60 | 60 | Yes | 0 |
| 6 | 94 | 70 | Yes | 10 |
| 7 | 110 | 26 | Yes | 5 |

- All 7 attacks are detected. Four of them are caught in their first hour.
- Attack 5 has an alarm in all 60 hours. Attack 4 has 86 of 94.
- Attack 3 is caught very late, 43 hours after it starts. Attack 7 has only 26 alarm hours out of 110, which fits step 2 (the replay hides it from `L_T1`).
- Attacks 2 and 7 have no labels in the original Dataset 2 file (`ATT_FLAG`). Our labels come from the 7 attack windows. Ridge still catches both, in the first hour for attack 2 and after 5 hours for attack 7.

Caution:
- 32 false alarm episodes in 21.9 weeks is about 1.5 per week. This is over the budget, even though Calibration was inside it.
- These are development scores. We looked at Dataset 2 while building the model, so they may look a bit better than the truth. The final numbers come from the Test set.

## 8. Look at the false alarms on Dataset 2

Why: we want to know where the false alarms come from. If they are right after an attack, they may be after-effects (the system is still recovering), not real mistakes. If they come from one sensor, that sensor may be the problem.

What the code does: for each false alarm hour, finds the sensor with the biggest scaled error (`top_fa`) and counts them. It loads the attack windows from `../data/attack_windows_dataset2.csv`, and puts each false alarm in a group: within 24 hours after an attack, within 24 hours before an attack, or far from any attack. Then it prints every false alarm with its time, group and top sensor.

In [9]:
scaled_d2 = err_d2.abs() / normal_size
top_fa = scaled_d2[false_alarm].idxmax(axis=1)
print(top_fa.value_counts())

attacks = pd.read_csv("../data/attack_windows_dataset2.csv", parse_dates=["real_start", "real_end"])

def nearest(t):
    after = (t - attacks["real_end"]).dt.total_seconds() / 3600
    before = (attacks["real_start"] - t).dt.total_seconds() / 3600
    if after[after > 0].min() <= 24:
        return "within 24h after an attack"
    if before[before > 0].min() <= 24:
        return "within 24h before an attack"
    return "far from any attack"

fa_times = d2.loc[false_alarm, "DATETIME"]
groups = fa_times.apply(nearest)
print(groups.value_counts())

print(pd.DataFrame({"time": fa_times, "group": groups, "top_sensor": top_fa}).to_string())

P_J422    16
F_PU6      6
P_J415     5
F_PU1      3
F_PU10     3
F_PU11     1
P_J307     1
P_J14      1
Name: count, dtype: int64
DATETIME
far from any attack            29
within 24h before an attack     6
within 24h after an attack      1
Name: count, dtype: int64
                    time                        group top_sensor
3    2016-07-04 03:00:00          far from any attack     P_J422
4    2016-07-04 04:00:00          far from any attack     P_J422
6    2016-07-04 06:00:00          far from any attack      F_PU1
72   2016-07-07 00:00:00          far from any attack      F_PU6
73   2016-07-07 01:00:00          far from any attack      F_PU6
371  2016-07-19 11:00:00          far from any attack     P_J422
440  2016-07-22 08:00:00          far from any attack     F_PU10
530  2016-07-26 02:00:00          far from any attack     P_J422
609  2016-07-29 09:00:00          far from any attack     P_J415
694  2016-08-01 22:00:00          far from any attack     P_J422
734  2016-08-03 14

**What this shows:**
- The false alarms are not after-effects. Only 1 is within 24 hours after an attack (2016-09-16 01:00, the hour after attack 1 ends). 29 are far from any attack, and 6 are within 24 hours before one.
- `P_J422` causes most of them (16), then `F_PU6` (6) and `P_J415` (5).
- The `F_PU6` false alarms come in pairs of hours in a row (2016-07-07 00:00 and 01:00, 2016-11-13 20:00 and 21:00, 2016-12-24 23:00 and 2016-12-25 00:00). Pump 6 runs only rarely, so its normal running hours look unusual to the model.
- Three false alarms are exactly 1 hour before an attack starts: 2016-09-26 10:00 (attack 2), 2016-10-29 18:00 (attack 4) and 2016-12-14 14:00 (attack 7). These might be the very start of the attack, just before the labelled window.
- 3 false alarms are in the first hours of Dataset 2 (2016-07-04 03:00, 04:00 and 06:00).

Caution: we did not check why `P_J422` makes so many false alarms. Its normal error is very small, about 0.001 (0.0011 in the step 4 table), so even a small change gives it a big scaled error.

## 9. Try again without `P_J422`

Why: `P_J422` causes most false alarms on both Calibration (step 6) and Dataset 2 (step 8). We check if the results get better when we leave it out of the score.

What the code does: builds the same score as step 5, but with only 27 sensors (`targets_v2`, all except `P_J422`). It sets a new threshold on Calibration with the same rule as step 6 (`threshold_v2`) and prints the Calibration alarms. Then it compares the Dataset 2 results and the delays before and after (`evaluate`, `delays`). The models are not refit, only the score changes.

In [10]:
targets_v2 = [c for c in cont_cols if c != "P_J422"]
normal_v2 = normal_size[targets_v2]

def hourly_score_v2(err):
    scaled = err[targets_v2].abs() / normal_v2
    top3 = np.sort(scaled.values, axis=1)[:, -3:]
    return pd.Series(top3.mean(axis=1), index=err.index)

score_cal_v2 = hourly_score_v2(err_cal)
score_d2_v2 = hourly_score_v2(err_d2)

start = score_cal_v2.quantile(0.99)
for t in np.sort(score_cal_v2.unique()):
    if t >= start and count_episodes(score_cal_v2 > t) <= max_episodes:
        threshold_v2 = t
        break

alarm_cal_v2 = score_cal_v2 > threshold_v2
print("Threshold:", round(threshold_v2, 3))
print("Calibration alarm hours:", alarm_cal_v2.sum(), "| episodes:", count_episodes(alarm_cal_v2))

def evaluate(score, thr):
    alarm = score > thr
    fa = alarm & (y == 0)
    return pd.Series({
        "ROC-AUC": roc_auc_score(y, score),
        "PR-AUC": average_precision_score(y, score),
        "Precision": precision_score(y, alarm),
        "Recall": recall_score(y, alarm),
        "F1": f1_score(y, alarm),
        "FA hours": fa.sum(),
        "FA episodes": count_episodes(fa),
    }).round(3)

def delays(score, thr):
    alarm = score > thr
    out = {}
    for a in range(1, 8):
        hits = alarm[d2["attack_id"] == a]
        out[a] = hits.values.argmax() if hits.any() else None
    return pd.Series(out)

print(pd.DataFrame({"before": evaluate(score_d2, threshold), "after": evaluate(score_d2_v2, threshold_v2)}))
print(pd.DataFrame({"delay_before": delays(score_d2, threshold), "delay_after": delays(score_d2_v2, threshold_v2)}))

Threshold: 6.031
Calibration alarm hours: 11 | episodes: 10
             before   after
ROC-AUC       0.866   0.865
PR-AUC        0.741   0.747
Precision     0.893   0.882
Recall        0.610   0.624
F1            0.725   0.731
FA hours     36.000  41.000
FA episodes  32.000  35.000
   delay_before  delay_after
1             0            0
2             0            0
3            43           43
4             0            0
5             0            0
6            10           10
7             5            5


**What this shows:**
- The new threshold is lower: 6.031 instead of 7.276. Calibration still has 11 alarm hours in 10 episodes.
- Dataset 2 stays about the same: F1 0.725 → 0.731, recall 0.610 → 0.624, PR-AUC 0.741 → 0.747, ROC-AUC 0.866 → 0.865.
- The false alarms actually go up: 36 → 41 hours and 32 → 35 episodes. With a lower threshold, other sensors' false alarms take the place of the `P_J422` ones.
- The delays for all 7 attacks are the same.

Decision: Calibration showed no difference (10 episodes either way), and we only make choices on Calibration, not on Dataset 2. So we keep all 28 sensors. The saved results below use the 28-sensor score and the 7.276 threshold.

## 10. Save the results

Why: the fusion step and the comparison with the other models need the Ridge scores, so we save them as files. We also save the models, so the Test set can be scored at the end without retraining.

What the code does: saves three files in `../models/Ridge/`:
- `ridge_scores.csv`: the hourly score and the alarm (1 or 0) for Validation, Calibration and Dataset 2, with `label` and `attack_id` for Dataset 2.
- `ridge_errors.csv`: the raw error of each of the 28 sensors for every hour in the same three blocks.
- `ridge_model.joblib`: the 28 fitted models, the best alphas, `normal_size`, the threshold and the feature list.

It then reads the scores file back and prints its shape, the alarm hours per block, and the files in the folder.

In [11]:
import os
import joblib

out = "../models/Ridge"
os.makedirs(out, exist_ok=True)

parts = []
for name, df, score in [("validation", val, score_val), ("calibration", cal, score_cal), ("dataset2", d2, score_d2)]:
    part = pd.DataFrame({"DATETIME": df["DATETIME"], "split": name,
                         "score": score, "alarm": (score > threshold).astype(int)})
    if name == "dataset2":
        part["label"] = df["label"]
        part["attack_id"] = df["attack_id"]
    parts.append(part)
pd.concat(parts, ignore_index=True).to_csv(f"{out}/ridge_scores.csv", index=False)

errs = []
for name, df, err in [("validation", val, err_val), ("calibration", cal, err_cal), ("dataset2", d2, err_d2)]:
    e = err.copy()
    e.insert(0, "DATETIME", df["DATETIME"])
    e.insert(1, "split", name)
    errs.append(e)
pd.concat(errs, ignore_index=True).to_csv(f"{out}/ridge_errors.csv", index=False)

joblib.dump({"models": models, "alpha": best_alpha, "normal_size": normal_size,
             "threshold": threshold, "features": features}, f"{out}/ridge_model.joblib")

check = pd.read_csv(f"{out}/ridge_scores.csv", parse_dates=["DATETIME"])
print(check.shape)
print(check.groupby("split")["alarm"].sum())
print(os.listdir(out))

(7634, 6)
split
calibration     11
dataset2       336
validation       6
Name: alarm, dtype: int64
['ridge_errors.csv', 'ridge_model.joblib', 'ridge_scores.csv']


**What this shows:**
- `ridge_scores.csv` has 7,634 rows (1,728 + 1,729 + 4,177) and 6 columns.
- Alarm hours: Validation 6 out of 1,728, Calibration 11 (as in step 6), Dataset 2 336.
- All three files are in `models/Ridge/`.

Caution: Validation was used to choose the alphas and `normal_size`, so its alarm count is not a fair test. It only shows the threshold is not far off on another normal block. Calibration's 11 alarm hours and 10 episodes meet the budget by design, since we chose the threshold on Calibration. The honest check on normal hours the threshold never saw is Dataset 2 outside the attack windows: 36 false alarm hours and 32 episodes in 21.9 weeks, which is over the episode budget.

## End of notebook

The full write-up is in `docs/ridge.md`.

Still open:
1. **Dataset 2 is over the false alarm budget.** It has 32 episodes in 21.9 weeks, about 1.5 per week, even though Calibration is inside the budget.
2. **A 2-hours-in-a-row alarm rule.** Only raising an alarm when 2 hours in a row are above the threshold could remove many of the single-hour false alarms. But this rule should be the same for every model, so it is a team decision.
3. **Attack 3 is detected 43 hours late.** It is caught, but very late.
4. **The Test set is scored only at the end**, after all choices and thresholds are frozen.